In [19]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sklearn.svm import LinearSVC
from scipy.sparse import hstack, csr_matrix

In [20]:
DATA_PATH = Path("accounts-bills.json")

with open(DATA_PATH, encoding="utf-8") as f:
    df = pd.json_normalize(json.load(f))

df["itemDescription"] = df["itemDescription"].fillna("")
df["itemName"] = df["itemName"].fillna("")
df["vendorId"] = df["vendorId"].astype(str)

df.shape

(4894, 7)

In [21]:
print("Records:", len(df))
print("Classes:", df["accountName"].nunique())
print("Vendors:", df["vendorId"].nunique())

vc = df["accountName"].value_counts()

print("Imbalance ratio:", vc.max() / vc.min())
print("Classes with <5 samples:", (vc < 5).sum())

Records: 4894
Classes: 103
Vendors: 337
Imbalance ratio: 1179.0
Classes with <5 samples: 34


In [22]:
vendor_map = df.groupby("vendorId")["accountName"]\
               .agg(lambda x: x.value_counts().index[0])

vendor_acc = (df["vendorId"].map(vendor_map) == df["accountName"]).mean()

print("Vendor-only accuracy:", vendor_acc)

Vendor-only accuracy: 0.7394769105026563


In [23]:
def make_text(row):
    name = str(row["itemName"]).lower().strip()
    desc = str(row["itemDescription"]).lower().strip()
    
    if desc and desc != name:
        return f"{name} {desc}"
    return name

df["text"] = df.apply(make_text, axis=1)

In [24]:
df["amount_log"] = np.log1p(np.abs(df["itemTotalAmount"]))

In [25]:
le = LabelEncoder()
y = le.fit_transform(df["accountName"])

In [ ]:
vc = df["accountName"].value_counts()

single_classes = vc[vc == 1].index

df_multi = df[~df["accountName"].isin(single_classes)]
df_single = df[df["accountName"].isin(single_classes)]

X_train, X_test, y_train, y_test = train_test_split(
    df_multi,
    le.transform(df_multi["accountName"]),
    test_size=0.2,
    stratify=df_multi["accountName"],
    random_state=42
)

X_train = pd.concat([X_train, df_single])
y_train = np.concatenate([y_train, le.transform(df_single["accountName"])])

In [27]:
tfidf_word = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    min_df=2,
    sublinear_tf=True
)

X_tr_word = tfidf_word.fit_transform(X_train["text"])
X_te_word = tfidf_word.transform(X_test["text"])

In [28]:
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=5000,
    min_df=2
)

X_tr_char = tfidf_char.fit_transform(X_train["text"])
X_te_char = tfidf_char.transform(X_test["text"])

In [29]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

X_tr_vendor = ohe.fit_transform(X_train[["vendorId"]])
X_te_vendor = ohe.transform(X_test[["vendorId"]])

In [30]:
scaler = StandardScaler()

amt_tr = scaler.fit_transform(
    np.column_stack([
        X_train["itemTotalAmount"],
        X_train["amount_log"]
    ])
)

amt_te = scaler.transform(
    np.column_stack([
        X_test["itemTotalAmount"],
        X_test["amount_log"]
    ])
)

X_tr_amt = csr_matrix(amt_tr)
X_te_amt = csr_matrix(amt_te)

In [31]:
X_train_final = hstack([
    X_tr_word,
    X_tr_char,
    X_tr_vendor,
    X_tr_amt
])

X_test_final = hstack([
    X_te_word,
    X_te_char,
    X_te_vendor,
    X_te_amt
])

X_train_final.shape

(3918, 13576)

### Fixing the imbalancing

In [ ]:
model = LinearSVC(
    C=5,
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    # verbose=1
)

model.fit(X_train_final, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",5
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo 

In [33]:
y_pred = model.predict(X_test_final)

acc = accuracy_score(y_test, y_pred)
f1w = f1_score(y_test, y_pred, average="weighted")
f1m = f1_score(y_test, y_pred, average="macro")

print("Accuracy:", acc)
print("F1 weighted:", f1w)
print("F1 macro:", f1m)

Accuracy: 0.9159836065573771
F1 weighted: 0.9174519917268195
F1 macro: 0.7734633581820562


In [34]:
y_pred = model.predict(X_test_final)

acc = accuracy_score(y_test, y_pred)
f1w = f1_score(y_test, y_pred, average="weighted")
f1m = f1_score(y_test, y_pred, average="macro")

print("Accuracy:", acc)
print("F1 weighted:", f1w)
print("F1 macro:", f1m)

Accuracy: 0.9159836065573771
F1 weighted: 0.9174519917268195
F1 macro: 0.7734633581820562


### Print errors

In [ ]:
# copy test dataframe
df_test = X_test.copy().reset_index(drop=True)

df_test["true"] = le.inverse_transform(y_test)
df_test["pred"] = le.inverse_transform(y_pred)

# correctness flag
df_test["is_correct"] = df_test["true"] == df_test["pred"]

errors = df_test[~df_test["is_correct"]]
correct = df_test[df_test["is_correct"]]

# metrics
error_rate = len(errors) / len(df_test)
accuracy = len(correct) / len(df_test)

print(f"Accuracy   : {accuracy:.4f}")
print(f"Error rate : {error_rate:.4f}")

print("\nAvg amount by correctness:")
print(df_test.groupby("is_correct")["itemTotalAmount"].mean())

print("\nSample errors:")
display(errors.head(10))


Accuracy   : 0.9160
Error rate : 0.0840

Avg amount by correctness:
is_correct
False    755884.698950
True     203530.750884
Name: itemTotalAmount, dtype: float64

Sample errors:


,vendorId,itemName,itemDescription,accountId,accountName,itemTotalAmount,_id.$oid,text,amount_log,true,pred,is_correct
3,EMf6OOZwrXTw0hi3eXjM,CMG Tote Bags,0324 CMG Tote Bags,46gT0Ab9FKwD2LHwpa4Q,612016 Collateral,642.20000,6659b2cd7f88e19b2ad71ddb,cmg tote bags 0324 cmg tote bags,6.466456,612016 Collateral,612023 Print Ads,False
17,fzS9iUgai9GGi5jhtSaQ,0424 Enteprise - Data,0424 Enteprise - Data,C0SB5vR0gCikdKjmT8Ep,611202 Online Subscription/Tool,1471.89000,6613bd53d24a7f8b003ff9b9,0424 enteprise - data,7.294982,611202 Online Subscription/Tool,132098 IC Clearing account,False
28,BkzdGW5tn3X0t53loSUQ,0126 Carousell Group - Bug Bounty Tracking ID:...,0126 Carousell Group - Bug Bounty Tracking ID:...,w5hjK0IooxdDp9Feeeuy,132098 IC Clearing account - Paid on Behalf,50.00000,69703a04c1e9794ae68d9166,0126 carousell group - bug bounty tracking id:...,3.931826,132098 IC Clearing account - Paid on Behalf,611203 Product Platform,False
31,uJjiPZ9Zhj8nPwpASydV,0226 - WhatsApp Monthly Fee Contract,Whatsapp monthly phone purchase,C0SB5vR0gCikdKjmT8Ep,611202 Online Subscription/Tool,38.00000,69840318ff33e42db83de263,0226 - whatsapp monthly fee contract whatsapp ...,3.663562,611202 Online Subscription/Tool,134001 Prepaid Operating Expense,False
38,KcLS8RMd3UrRlSKbImS5,0524 Sendbird 2024,0524 Sendbird 2024,C0SB5vR0gCikdKjmT8Ep,611202 Online Subscription/Tool,26578.00000,66389bec79be3d61295d3ff8,0524 sendbird 2024,10.187877,611202 Online Subscription/Tool,134001 Prepaid Operating Expense,False
73,8y2BQtMF74TubxeuYeAQ,1225 Re: Refund | Scube Gift Pte Ltd (INV-02-1...,1225 Re: Refund | Scube Gift Pte Ltd (INV-02-1...,AS7kZeVZgNNPi7UQOSKD,231101 Customer Prepayment,4180.00000,6941541090dd6756112ff6d0,1225 re: refund | scube gift pte ltd (inv-02-1...,8.338306,231101 Customer Prepayment,222003 Customer Prepayment,False
75,7dkV8PetzeNAZYIiG7Br,0225 Sonarcloud is code analysis tool for both...,0225 Sonarcloud is code analysis tool for both...,w5hjK0IooxdDp9Feeeuy,132098 IC Clearing account,1518.80652,67bbe58dd0f673d72c64f92d,0225 sonarcloud is code analysis tool for both...,7.326338,132098 IC Clearing account,611202 Online Subscription/Tool,False
88,r8GpizJBxnWJC9IwPW2p,0825 Atlassian Monthly payment subscription,0825 Atlassian Monthly payment subscription,C0SB5vR0gCikdKjmT8Ep,611202 Online Subscription/Tool,6690.13000,68884f1be1ddb3ecdfc42ff2,0825 atlassian monthly payment subscription,8.808538,611202 Online Subscription/Tool,132098 IC Clearing account,False
103,lL1pcuEf3q6ufBVg2R75,1124-1125 Annual Bill for Docker-2025,1124-1125 Annual Bill for Docker,vNCPOlaDL0NMQTi5ca5Q,134001 Prepaid Operating Expense,8513.52360,674d8fcac7572af238539790,1124-1125 annual bill for docker-2025 1124-112...,9.049529,134001 Prepaid Operating Expense,611202 Online Subscription/Tool,False
141,wTtaNgtrU4MLiDH4pxAD,Truescope media monitoring,0225-0126 Truescope media monitoring,w5hjK0IooxdDp9Feeeuy,132098 IC Clearing account,15696.00000,67a05def7dad2c1b94dafa8a,truescope media monitoring 0225-0126 truescope...,9.661225,132098 IC Clearing account,134001 Prepaid Operating Expense,False


### Save Model

In [36]:
import os
import joblib

# create directory
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# save all artifacts
joblib.dump(model, f"{MODEL_DIR}/expense_classifier.pkl")
joblib.dump(tfidf_word, f"{MODEL_DIR}/tfidf_word.pkl")
joblib.dump(tfidf_char, f"{MODEL_DIR}/tfidf_char.pkl")
joblib.dump(ohe, f"{MODEL_DIR}/ohe_vendor.pkl")
joblib.dump(scaler, f"{MODEL_DIR}/amount_scaler.pkl")
joblib.dump(le, f"{MODEL_DIR}/label_encoder.pkl")

print("All models saved to:", MODEL_DIR)

All models saved to: models
